
# FFT and harmonic analysis of human-blood rheometer data

This notebook performs frequency-domain analysis of the **JUN18**, **APR18**, or **DEC17** ARES-G2 blood-rheology datasets using `scipy.fft`.

It is designed especially for **LAOS** data, because LAOS has a known periodic forcing frequency and therefore gives physically meaningful harmonic information. The same FFT utilities can also be used on other signals, but the interpretation of "harmonics" is strongest for periodic experiments.

### Outputs

The notebook produces:

- time-domain stress / strain / shear-rate plots;
- one-sided FFT amplitude spectra;
- relative spectral-power plots;
- dominant-frequency peak tables;
- harmonic amplitudes, phases, and ratios;
- \(I_3/I_1\) and total harmonic distortion (THD);
- individual harmonic-component reconstructions;
- original-vs-harmonic reconstructed stress;
- stress / strain / shear-rate spectral comparison;
- an FFT-based estimate of the fundamental complex modulus;
- cross-experiment comparison across strain amplitudes and angular frequencies;
- CSV exports of spectra, harmonic tables, and batch summaries.

### Dataset references used by this project

- JUN18: Armstrong & Horner, *Rheology Data of Human Blood JUN18*, Mendeley Data, DOI `10.17632/948ffnypjs.1`
- DEC17: Armstrong, *DEC17 ARESG2 Human Blood Rheology: SS, SAOS, LAOS, STEPS, Etc.*, DOI `10.17632/d43r5ydpx8.1`
- APR18: Armstrong & Horner, *APR18 SS SAOS LAOS & UDLAOS of Human Blood*, DOI `10.17632/6jjryg2f8v.1`

> **Important:** FFT is a global frequency transform. It tells you what frequencies are present over the selected time window, but not precisely *when* a frequency appeared. For local transition/changepoint timing, STFT or CWT is more appropriate.


In [ ]:

# Core imports
from pathlib import Path
import os
import re
import math
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.fft import rfft, rfftfreq, irfft
from scipy import signal

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
import scipy
print("SciPy:", scipy.__version__)



## 1. Configuration

For harmonic analysis, start with a **LAOS** sheet.

The notebook will automatically search for the workbook under:

1. `data/` beneath the notebook directory,
2. the notebook/current working directory,
3. `/mnt/data` when running in the ChatGPT environment.

If automatic discovery fails, set `DATA_FILE` explicitly.


In [ ]:

# ---------------- USER SETTINGS ----------------

DATASET_KEY = "DEC17"          # "JUN18", "APR18", or "DEC17"

# Optional explicit path. Leave as None to auto-search.
# Examples:
# DATA_FILE = Path(r"C:\path\to\JUN18.xlsx")
# DATA_FILE = Path(r"C:\path\to\APR18.xlsx")
DATA_FILE = Path(r"C:\Users\vannd\Documents\UniMelb\0_2026S1_Captstone\Capstone_2026S2\MAST90106-Data-Science-Project---Group-35\data\UDEL Blood DEC17 SS SAOS LAOS STEPS RAMPS HELTON.xlsx")
# DATA_FILE = None

# For Excel workbooks, leave None to automatically select a LAOS sheet.
SHEET_NAME = None

# Select an extracted experiment/block after the experiment table is displayed.
# Matching is case-insensitive. The first match is used.
EXPERIMENT_QUERY = "omega=1; Gamma=10"

# Signal used for the main FFT.
SIGNAL_COLUMN = "stress"      # "stress", "strain", "shear_rate", or "viscosity" if available

# Use the last N complete cycles when a forcing frequency can be parsed.
# This strongly reduces spectral leakage for LAOS.
N_CYCLES = 5

# Number of harmonics to quantify.
N_HARMONICS = 9

# Harmonics plotted individually.
HARMONICS_TO_SHOW = [1, 3, 5, 7, 9]

# "boxcar" is preferred when using an integer number of LAOS cycles.
# Use "hann" for a generic non-integer/non-periodic window.
FFT_WINDOW = "boxcar"        # "boxcar", "hann", or None

# Detrending before FFT.
DETREND = "constant"         # "constant", "linear", or None

# Spectrum display. None chooses a useful limit from f0 and Nyquist.
MAX_PLOT_FREQ_HZ = None

# Peak detection threshold relative to the maximum non-DC FFT amplitude.
PEAK_PROMINENCE_FRACTION = 0.02

# Export folder
RESULTS_DIR = Path("results_fft")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Locate the selected dataset

In [ ]:

DATASET_PATTERNS = {
    "JUN18": [
        "*JUN18*.xlsx",
        "*JUN18*.xls",
        "*JUN18*.zip",
        "*Rheology data of human blood JUN18*.zip",
    ],
    "APR18": [
        "*APR18*.xlsx",
        "*APR18*.xls",
        "*APR18*.zip",
        "*APR*18*.xlsx",
        "*APR*18*.zip",
    ],
    "DEC17": [
        "*DEC17*.xlsx",
        "*DEC17*.xls",
        "*DEC17*.zip",
        "*UDEL Blood DEC17*.xlsx",
    ],
}

def search_roots():
    roots = [
        Path.cwd() / "data",
        Path.cwd(),
        Path("/mnt/data"),
    ]
    # Keep only unique existing roots.
    unique = []
    seen = set()
    for root in roots:
        try:
            resolved = root.resolve()
        except Exception:
            resolved = root
        key = str(resolved)
        if root.exists() and key not in seen:
            unique.append(root)
            seen.add(key)
    return unique

def find_first_match(patterns):
    matches = []
    for root in search_roots():
        for pattern in patterns:
            try:
                matches.extend(root.rglob(pattern))
            except Exception:
                pass
    # Prefer direct Excel files over ZIP archives.
    matches = sorted(
        set(matches),
        key=lambda p: (p.suffix.lower() == ".zip", len(str(p)), str(p).lower())
    )
    return matches[0] if matches else None

def extract_excel_from_zip(zip_path, dataset_key):
    cache = Path(".fft_dataset_cache") / dataset_key
    cache.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as zf:
        members = [
            name for name in zf.namelist()
            if name.lower().endswith((".xlsx", ".xls"))
        ]
        if not members:
            raise FileNotFoundError(f"No Excel workbook found inside {zip_path}")
        member = members[0]
        target_name = Path(member).name
        target = cache / target_name
        if not target.exists():
            with zf.open(member) as src, open(target, "wb") as dst:
                dst.write(src.read())
        return target

def resolve_dataset_file(dataset_key, explicit_path=None):
    if explicit_path is not None:
        p = Path(explicit_path)
        if not p.exists():
            raise FileNotFoundError(f"DATA_FILE does not exist: {p}")
    else:
        patterns = DATASET_PATTERNS.get(dataset_key.upper())
        if patterns is None:
            raise ValueError(f"Unknown DATASET_KEY={dataset_key!r}")
        p = find_first_match(patterns)
        if p is None:
            raise FileNotFoundError(
                f"Could not automatically locate {dataset_key}. "
                "Put the workbook under data/ or set DATA_FILE explicitly."
            )

    if p.suffix.lower() == ".zip":
        p = extract_excel_from_zip(p, dataset_key.upper())

    return p

dataset_path = resolve_dataset_file(DATASET_KEY, DATA_FILE)
print("Dataset:", DATASET_KEY)
print("Resolved file:", dataset_path)


## 3. Inspect workbook sheets and choose a LAOS sheet

In [ ]:

def choose_sheet(workbook_path, requested=None):
    xl = pd.ExcelFile(workbook_path)
    sheets = xl.sheet_names

    print("Available sheets:")
    for i, name in enumerate(sheets):
        print(f"  [{i:02d}] {name}")

    if requested is not None:
        if requested not in sheets:
            raise ValueError(f"SHEET_NAME={requested!r} not found in workbook.")
        return requested

    # Prefer an exact LAOS sheet, then LAOS t0, then any sheet containing LAOS.
    preferences = ["LAOS", "LAOS t0", "LAOS T0"]
    for pref in preferences:
        for s in sheets:
            if s.strip().lower() == pref.lower():
                return s

    laos = [s for s in sheets if "laos" in s.lower() and "udlaos" not in s.lower()]
    if laos:
        return laos[0]

    raise ValueError(
        "No LAOS sheet could be selected automatically. "
        "Set SHEET_NAME manually."
    )

selected_sheet = choose_sheet(dataset_path, SHEET_NAME)
print("\nSelected sheet:", selected_sheet)



## 4. Extract side-by-side ARES-G2 experiments

The original workbooks often store many experiments **horizontally** in the same sheet rather than as one tidy table.

The helper below detects repeated `Step time` columns, extracts each experiment, standardizes the column names, converts strain `%` to dimensionless strain, and derives shear rate from strain if it is not already present.


In [ ]:

def _norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def detect_header_row(raw, max_rows=30):
    best_count = -1
    best_row = None
    for i in range(min(max_rows, len(raw))):
        count = sum(
            ("step time" in _norm_text(v)) or (_norm_text(v) == "time")
            for v in raw.iloc[i].tolist()
        )
        if count > best_count:
            best_count = count
            best_row = i

    if best_count <= 0:
        raise ValueError("Could not locate repeated 'Step time' / 'Time' headers.")
    return best_row, best_count

def parse_protocol_label(label):
    text = str(label)

    def number_after(pattern):
        m = re.search(pattern, text, flags=re.IGNORECASE)
        return float(m.group(1)) if m else np.nan

    omega = number_after(r"omega\s*=\s*([-+0-9.eE]+)")
    gamma = number_after(r"(?:gamma|amp)\s*=\s*([-+0-9.eE]+)")
    t_half = number_after(r"t1/2\s*=\s*([-+0-9.eE]+)")

    f0 = omega / (2 * np.pi) if np.isfinite(omega) else np.nan

    return {
        "label": text,
        "omega_rad_s": omega,
        "f0_hz": f0,
        "gamma": gamma,
        "t_half_s": t_half,
    }

def extract_aresg2_blocks(raw):
    header_row, n_blocks_detected = detect_header_row(raw)

    starts = [
        j for j, value in enumerate(raw.iloc[header_row].tolist())
        if ("step time" in _norm_text(value)) or (_norm_text(value) == "time")
    ]

    blocks = {}

    for block_i, start in enumerate(starts):
        next_start = starts[block_i + 1] if block_i + 1 < len(starts) else raw.shape[1]

        # Usually experiments occupy 2–5 columns with an empty separator.
        # Restrict the search width to avoid accidentally absorbing another block.
        end = min(next_start, start + 8)

        protocol_label = (
            raw.iat[header_row - 1, start]
            if header_row > 0 and pd.notna(raw.iat[header_row - 1, start])
            else f"experiment_{block_i+1}"
        )

        unit_row = header_row + 1
        units = {
            j: _norm_text(raw.iat[unit_row, j])
            for j in range(start, end)
            if unit_row < len(raw)
        }

        mapping = {}
        for j in range(start, end):
            h = _norm_text(raw.iat[header_row, j])

            if ("step time" in h) or h == "time":
                mapping["time"] = j
            elif "stress" in h and "normal" not in h:
                mapping.setdefault("stress", j)
            elif "strain" in h:
                # Prefer the first strain column for LAOS.
                mapping.setdefault("strain", j)
            elif "shear" in h and "stress" not in h:
                mapping.setdefault("shear_rate", j)
            elif "viscos" in h:
                mapping.setdefault("viscosity", j)

        if "time" not in mapping:
            continue

        # Some ARES-G2 sheets contain a missing/blank "Stress" header
        # even though the stress data column is present (e.g. certain
        # JUN18 LAOS blocks). Infer stress from the column immediately
        # after time when it is otherwise unmapped and the strain column
        # appears later in the block.
        if "stress" not in mapping:
            candidate = start + 1
            occupied = set(mapping.values())
            if (
                candidate < end
                and candidate not in occupied
                and ("strain" not in mapping or candidate < mapping["strain"])
            ):
                numeric_candidate = pd.to_numeric(
                    raw.iloc[header_row + 2 :, candidate],
                    errors="coerce"
                )
                if numeric_candidate.notna().sum() >= 3:
                    mapping["stress"] = candidate

        data = {}
        for canonical_name, col_idx in mapping.items():
            values = pd.to_numeric(
                raw.iloc[header_row + 2 :, col_idx],
                errors="coerce"
            ).to_numpy(dtype=float)

            if canonical_name == "strain" and units.get(col_idx, "") == "%":
                values = values / 100.0

            data[canonical_name] = values

        block = pd.DataFrame(data)
        block = block.replace([np.inf, -np.inf], np.nan)
        block = block.dropna(subset=["time"]).copy()
        block = block.sort_values("time")
        block = block.drop_duplicates(subset=["time"], keep="first")
        block = block.reset_index(drop=True)

        # Derive shear rate if strain is available but shear rate is absent.
        if "strain" in block.columns and "shear_rate" not in block.columns:
            good = np.isfinite(block["time"]) & np.isfinite(block["strain"])
            if good.sum() >= 3:
                t = block.loc[good, "time"].to_numpy()
                gamma = block.loc[good, "strain"].to_numpy()
                block.loc[good, "shear_rate"] = np.gradient(gamma, t)

        # Ensure unique dictionary key.
        key = str(protocol_label)
        if key in blocks:
            key = f"{key} [{block_i+1}]"

        blocks[key] = block

    return blocks, header_row, n_blocks_detected

raw_sheet = pd.read_excel(dataset_path, sheet_name=selected_sheet, header=None)
blocks, detected_header_row, detected_count = extract_aresg2_blocks(raw_sheet)

print(f"Detected header row: {detected_header_row}")
print(f"Detected experiments: {len(blocks)}")

experiment_rows = []
for i, (label, df) in enumerate(blocks.items()):
    info = parse_protocol_label(label)
    experiment_rows.append({
        "index": i,
        "experiment": label,
        "rows": len(df),
        "omega_rad_s": info["omega_rad_s"],
        "f0_hz": info["f0_hz"],
        "gamma": info["gamma"],
        "columns": ", ".join(df.columns),
    })

experiment_table = pd.DataFrame(experiment_rows)
display(experiment_table)


## 5. Select one experiment for detailed FFT analysis

In [ ]:

def choose_experiment(blocks, query=None):
    labels = list(blocks.keys())

    if not labels:
        raise ValueError("No experiments were extracted.")

    if query:
        matches = [label for label in labels if query.lower() in label.lower()]
        if matches:
            return matches[0]

        warnings.warn(
            f"No experiment matched {query!r}; using the first extracted experiment."
        )

    return labels[0]

selected_experiment = choose_experiment(blocks, EXPERIMENT_QUERY)
block = blocks[selected_experiment].copy()
protocol = parse_protocol_label(selected_experiment)

print("Selected experiment:", selected_experiment)
print("Protocol:", protocol)
print("\nColumns:", list(block.columns))
display(block.head())
display(block.tail())



## 6. Build a uniform time grid

`scipy.fft` assumes uniformly sampled data.

ARES-G2 timestamps can contain small timing variations, so the signal is linearly interpolated onto a uniform grid before the FFT.

When the LAOS forcing frequency \(f_0=\omega/(2\pi)\) is known, the notebook selects the **last complete integer number of cycles**. This is important because it places the fundamental and its harmonics close to exact FFT bins and substantially reduces spectral leakage.


In [ ]:

def clean_numeric_block(df):
    out = df.copy()
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["time"])
    out = out.sort_values("time").drop_duplicates("time").reset_index(drop=True)
    return out

def uniformize_block(df, f0=None, n_cycles=None):
    df = clean_numeric_block(df)

    t_all = df["time"].to_numpy(dtype=float)
    if len(t_all) < 4:
        raise ValueError("Not enough valid time samples.")

    dt_median = np.median(np.diff(t_all))
    if not np.isfinite(dt_median) or dt_median <= 0:
        raise ValueError("Invalid time spacing.")

    t_start_data = t_all[0]
    t_end_data = t_all[-1]
    available_duration = t_end_data - t_start_data

    used_cycles = None

    if f0 is not None and np.isfinite(f0) and f0 > 0 and n_cycles:
        max_complete_cycles = int(np.floor(available_duration * f0))

        if max_complete_cycles >= 1:
            used_cycles = min(int(n_cycles), max_complete_cycles)
            duration = used_cycles / f0

            # End at the last available sample and move back an integer
            # number of cycles.
            t0 = t_end_data - duration

            # Choose N so that T = N*dt exactly on the new grid.
            # This makes f0 an integer FFT-bin multiple.
            n_uniform = max(8, int(round(duration / dt_median)))
            dt_uniform = duration / n_uniform
            t_uniform = t0 + np.arange(n_uniform) * dt_uniform
        else:
            warnings.warn("Less than one complete forcing cycle is available.")
            t_uniform = np.arange(
                t_start_data, t_end_data + 0.5 * dt_median, dt_median
            )
    else:
        t_uniform = np.arange(
            t_start_data, t_end_data + 0.5 * dt_median, dt_median
        )

    uniform = pd.DataFrame({"time": t_uniform})

    for col in df.columns:
        if col == "time":
            continue

        valid = np.isfinite(df["time"].to_numpy()) & np.isfinite(
            pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float)
        )

        if valid.sum() < 2:
            continue

        t_src = df.loc[valid, "time"].to_numpy(dtype=float)
        y_src = pd.to_numeric(df.loc[valid, col], errors="coerce").to_numpy(dtype=float)
        uniform[col] = np.interp(t_uniform, t_src, y_src)

    return uniform, used_cycles

f0_protocol = protocol["f0_hz"]
f0_for_analysis = f0_protocol if np.isfinite(f0_protocol) else None

uniform_block, cycles_used = uniformize_block(
    block,
    f0=f0_for_analysis,
    n_cycles=N_CYCLES
)

if SIGNAL_COLUMN not in uniform_block.columns:
    raise ValueError(
        f"SIGNAL_COLUMN={SIGNAL_COLUMN!r} is not available. "
        f"Available columns: {list(uniform_block.columns)}"
    )

t = uniform_block["time"].to_numpy()
y = uniform_block[SIGNAL_COLUMN].to_numpy()

dt = np.mean(np.diff(t))
fs = 1.0 / dt
nyquist = fs / 2
duration = len(t) * dt
df_hz = 1.0 / duration

sampling_summary = pd.DataFrame([{
    "selected_experiment": selected_experiment,
    "signal": SIGNAL_COLUMN,
    "samples": len(t),
    "cycles_used": cycles_used,
    "duration_s": duration,
    "dt_s": dt,
    "fs_hz": fs,
    "nyquist_hz": nyquist,
    "frequency_resolution_hz": df_hz,
    "protocol_f0_hz": f0_for_analysis,
}])

display(sampling_summary)


## 7. Time-domain signal

In [ ]:

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, y)
ax.set_title(f"{DATASET_KEY} | {selected_experiment} | {SIGNAL_COLUMN}")
ax.set_xlabel("Time (s)")
ax.set_ylabel(SIGNAL_COLUMN)
ax.grid(True, alpha=0.25)
plt.show()



## 8. FFT with `scipy.fft`

For a real-valued signal, `rfft` computes only the non-negative-frequency half of the spectrum and `rfftfreq` gives the corresponding frequency bins.

The amplitude scaling below corrects for the coherent gain of the selected window.


In [ ]:

def get_window(name, n):
    if name is None:
        return np.ones(n)

    name = str(name).lower()
    if name in {"boxcar", "rectangular", "rect"}:
        return signal.windows.boxcar(n)
    if name in {"hann", "hanning"}:
        return signal.windows.hann(n, sym=False)

    raise ValueError(f"Unsupported window: {name!r}")

def compute_fft(t, y, window="boxcar", detrend="constant"):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)

    good = np.isfinite(t) & np.isfinite(y)
    t = t[good]
    y = y[good]

    if len(t) < 4:
        raise ValueError("Not enough samples for FFT.")

    dt = np.mean(np.diff(t))
    fs = 1.0 / dt
    n = len(y)

    if detrend is None:
        y_proc = y.copy()
    elif detrend in {"constant", "linear"}:
        y_proc = signal.detrend(y, type=detrend)
    else:
        raise ValueError("DETREND must be 'constant', 'linear', or None.")

    w = get_window(window, n)
    coherent_gain = np.mean(w)

    Y = rfft(y_proc * w)
    freq = rfftfreq(n, d=dt)

    amplitude = np.abs(Y) / (n * coherent_gain)

    # Convert to a one-sided amplitude spectrum.
    if n > 1:
        amplitude[1:-1] *= 2
        if n % 2 == 1:
            amplitude[-1] *= 2

    phase = np.angle(Y)

    power_proxy = amplitude ** 2
    denom = power_proxy[1:].sum()
    relative_power_pct = (
        100 * power_proxy / denom if denom > 0
        else np.zeros_like(power_proxy)
    )

    return {
        "t": t,
        "y_original": y,
        "y_processed": y_proc,
        "dt": dt,
        "fs": fs,
        "n": n,
        "freq": freq,
        "fft_complex": Y,
        "amplitude": amplitude,
        "phase_rad": phase,
        "relative_power_pct": relative_power_pct,
        "window": window,
        "detrend": detrend,
    }

fft_result = compute_fft(
    t,
    y,
    window=FFT_WINDOW,
    detrend=DETREND
)

freq = fft_result["freq"]
amplitude = fft_result["amplitude"]
relative_power_pct = fft_result["relative_power_pct"]

print("FFT complete.")
print("Frequency resolution:", freq[1] - freq[0], "Hz")
print("Nyquist:", freq[-1], "Hz")


In [ ]:

def choose_plot_max_frequency(result, f0=None, user_max=None, n_harmonics=9):
    nyq = result["freq"][-1]

    if user_max is not None:
        return min(float(user_max), nyq)

    if f0 is not None and np.isfinite(f0) and f0 > 0:
        return min(nyq, max(2.0, 1.2 * n_harmonics * f0))

    return min(nyq, 5.0)

plot_max_hz = choose_plot_max_frequency(
    fft_result,
    f0=f0_for_analysis,
    user_max=MAX_PLOT_FREQ_HZ,
    n_harmonics=N_HARMONICS
)

mask = freq <= plot_max_hz

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(freq[mask], amplitude[mask])
ax.set_title(f"One-sided FFT amplitude spectrum | {SIGNAL_COLUMN}")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Amplitude")
ax.grid(True, alpha=0.25)

if f0_for_analysis is not None:
    for h in range(1, N_HARMONICS + 1):
        fh = h * f0_for_analysis
        if fh <= plot_max_hz:
            ax.axvline(fh, linestyle="--", alpha=0.25)

plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(freq[mask], relative_power_pct[mask])
ax.set_title(f"Relative FFT spectral power | {SIGNAL_COLUMN}")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Relative power (%)")
ax.grid(True, alpha=0.25)
plt.show()


## 9. Dominant spectral peaks

In [ ]:

def dominant_peak_table(result, max_frequency=None, prominence_fraction=0.02, top_n=15):
    f = result["freq"]
    a = result["amplitude"]

    valid = f > 0
    if max_frequency is not None:
        valid &= f <= max_frequency

    f_sub = f[valid]
    a_sub = a[valid]

    if len(a_sub) < 3 or np.nanmax(a_sub) <= 0:
        return pd.DataFrame()

    prominence = prominence_fraction * np.nanmax(a_sub)
    peak_idx, props = signal.find_peaks(a_sub, prominence=prominence)

    if len(peak_idx) == 0:
        return pd.DataFrame()

    table = pd.DataFrame({
        "frequency_hz": f_sub[peak_idx],
        "amplitude": a_sub[peak_idx],
        "prominence": props["prominences"],
    })

    table["period_s"] = 1.0 / table["frequency_hz"]
    table = table.sort_values("amplitude", ascending=False).head(top_n).reset_index(drop=True)
    return table

peak_table = dominant_peak_table(
    fft_result,
    max_frequency=plot_max_hz,
    prominence_fraction=PEAK_PROMINENCE_FRACTION,
    top_n=15
)

display(peak_table)



## 10. Harmonic table

For LAOS with known \(\omega\),

\[
f_0 = \frac{\omega}{2\pi}.
\]

The \(n\)-th harmonic is expected near

\[
f_n = n f_0.
\]

The notebook searches the nearest FFT bins around each target harmonic and reports:

- measured harmonic frequency;
- amplitude;
- phase;
- amplitude relative to the first harmonic;
- \(I_3/I_1\);
- THD;
- odd-harmonic THD.

For nonlinear LAOS, the third and higher odd harmonics are particularly useful indicators of departure from a purely sinusoidal response.


In [ ]:

def estimate_fundamental_from_fft(result, min_frequency=None, max_frequency=None):
    f = result["freq"]
    a = result["amplitude"]

    if min_frequency is None:
        min_frequency = max(f[1] if len(f) > 1 else 0, 1.0 / (result["n"] * result["dt"]))

    valid = f >= min_frequency
    if max_frequency is not None:
        valid &= f <= max_frequency

    if not np.any(valid):
        raise ValueError("No FFT frequencies available in the selected search range.")

    idx_local = np.argmax(a[valid])
    idx_global = np.where(valid)[0][idx_local]
    return float(f[idx_global])

if f0_for_analysis is None:
    f0_for_analysis = estimate_fundamental_from_fft(fft_result)
    print("Protocol frequency unavailable; estimated f0 from FFT:", f0_for_analysis, "Hz")
else:
    print("Using protocol f0:", f0_for_analysis, "Hz")

def harmonic_table(result, f0, n_harmonics=9, search_bins=2):
    f = result["freq"]
    a = result["amplitude"]
    phase = result["phase_rad"]

    rows = []

    for h in range(1, n_harmonics + 1):
        target = h * f0

        if target > f[-1]:
            break

        center = int(np.argmin(np.abs(f - target)))
        lo = max(1, center - search_bins)
        hi = min(len(f), center + search_bins + 1)

        local_idx = lo + int(np.argmax(a[lo:hi]))

        rows.append({
            "harmonic": h,
            "target_frequency_hz": target,
            "measured_frequency_hz": f[local_idx],
            "frequency_error_hz": f[local_idx] - target,
            "amplitude": a[local_idx],
            "phase_deg": np.degrees(phase[local_idx]),
            "fft_bin": local_idx,
        })

    table = pd.DataFrame(rows)

    if table.empty:
        return table

    a1 = table.loc[table["harmonic"] == 1, "amplitude"].iloc[0]
    table["amplitude_over_H1"] = table["amplitude"] / a1 if a1 > 0 else np.nan
    table["amplitude_over_H1_pct"] = 100 * table["amplitude_over_H1"]

    return table

harmonics = harmonic_table(
    fft_result,
    f0=f0_for_analysis,
    n_harmonics=N_HARMONICS
)

display(harmonics)

if not harmonics.empty:
    H1 = harmonics.loc[harmonics["harmonic"] == 1, "amplitude"].iloc[0]

    higher = harmonics.loc[harmonics["harmonic"] >= 2, "amplitude"].to_numpy()
    odd_higher = harmonics.loc[
        (harmonics["harmonic"] >= 3) & (harmonics["harmonic"] % 2 == 1),
        "amplitude"
    ].to_numpy()

    thd = np.sqrt(np.sum(higher ** 2)) / H1 if H1 > 0 else np.nan
    odd_thd = np.sqrt(np.sum(odd_higher ** 2)) / H1 if H1 > 0 else np.nan

    h3_rows = harmonics.loc[harmonics["harmonic"] == 3, "amplitude_over_H1"]
    i3_i1 = h3_rows.iloc[0] if len(h3_rows) else np.nan

    metrics = pd.DataFrame([{
        "f0_hz": f0_for_analysis,
        "H1_amplitude": H1,
        "I3_over_I1": i3_i1,
        "THD": thd,
        "THD_pct": 100 * thd,
        "odd_THD": odd_thd,
        "odd_THD_pct": 100 * odd_thd,
    }])

    display(metrics)


In [ ]:

if not harmonics.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(
        harmonics["harmonic"].astype(str),
        harmonics["amplitude_over_H1_pct"]
    )
    ax.set_title("Harmonic amplitudes relative to H1")
    ax.set_xlabel("Harmonic number")
    ax.set_ylabel("Amplitude / H1 (%)")
    ax.grid(True, axis="y", alpha=0.25)
    plt.show()


## 11. Individual harmonic components and harmonic reconstruction

In [ ]:

def unwindowed_fft_for_reconstruction(t, y, detrend="constant"):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)

    if detrend is None:
        y_proc = y.copy()
        baseline = 0.0
    elif detrend == "constant":
        baseline = np.mean(y)
        y_proc = y - baseline
    elif detrend == "linear":
        y_proc = signal.detrend(y, type="linear")
        baseline = np.mean(y - y_proc)
    else:
        raise ValueError("Unsupported detrend option.")

    dt = np.mean(np.diff(t))
    Y = rfft(y_proc)
    f = rfftfreq(len(y_proc), d=dt)

    return y_proc, baseline, Y, f

def reconstruct_harmonics(t, y, f0, harmonics_to_keep, detrend="constant"):
    y_proc, baseline, Y, f = unwindowed_fft_for_reconstruction(t, y, detrend=detrend)

    Y_keep = np.zeros_like(Y, dtype=complex)
    component_series = {}

    for h in harmonics_to_keep:
        target = h * f0
        if target > f[-1]:
            continue

        idx = int(np.argmin(np.abs(f - target)))

        Y_h = np.zeros_like(Y, dtype=complex)
        Y_h[idx] = Y[idx]

        component = irfft(Y_h, n=len(y_proc))
        component_series[h] = component
        Y_keep[idx] = Y[idx]

    reconstruction = irfft(Y_keep, n=len(y_proc)) + baseline

    return reconstruction, component_series, f

valid_harmonics_to_show = [
    h for h in HARMONICS_TO_SHOW
    if h * f0_for_analysis <= nyquist
]

reconstructed, harmonic_components, reconstruction_freq = reconstruct_harmonics(
    t,
    y,
    f0=f0_for_analysis,
    harmonics_to_keep=valid_harmonics_to_show,
    detrend=DETREND
)

# Plot each component on its own figure.
for h in valid_harmonics_to_show:
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(t, harmonic_components[h])
    ax.set_title(f"H{h} component | {h*f0_for_analysis:.4f} Hz")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel(SIGNAL_COLUMN)
    ax.grid(True, alpha=0.25)
    plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, y, label="Measured")
ax.plot(t, reconstructed, label=f"Reconstruction {valid_harmonics_to_show}")
ax.set_title("Measured signal vs selected-harmonic reconstruction")
ax.set_xlabel("Time (s)")
ax.set_ylabel(SIGNAL_COLUMN)
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

residual = y - reconstructed
rmse = np.sqrt(np.mean(residual ** 2))
ss_res = np.sum(residual ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r2_reconstruction = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

reconstruction_metrics = pd.DataFrame([{
    "harmonics_used": str(valid_harmonics_to_show),
    "RMSE": rmse,
    "R2": r2_reconstruction,
}])

display(reconstruction_metrics)



## 12. Compare stress, strain, and shear-rate spectra

The spectra are normalized to their own largest non-DC amplitude so that signals with different physical units can be compared by spectral shape.

This does **not** mean their raw amplitudes are directly comparable.


In [ ]:

available_signals = [
    col for col in ["stress", "strain", "shear_rate", "viscosity"]
    if col in uniform_block.columns
]

fig, ax = plt.subplots(figsize=(12, 5))

spectral_results = {}

for col in available_signals:
    res = compute_fft(
        uniform_block["time"].to_numpy(),
        uniform_block[col].to_numpy(),
        window=FFT_WINDOW,
        detrend=DETREND
    )
    spectral_results[col] = res

    f = res["freq"]
    a = res["amplitude"].copy()

    non_dc_max = np.max(a[1:]) if len(a) > 1 else np.max(a)
    a_norm = a / non_dc_max if non_dc_max > 0 else a

    m = f <= plot_max_hz
    ax.plot(f[m], a_norm[m], label=col)

ax.set_title("Normalized FFT spectrum comparison")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Normalized amplitude")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()



## 13. FFT-based fundamental stress/strain response

For periodic LAOS data containing both stress and strain, the ratio of the complex Fourier coefficients at the first harmonic gives an estimate of the fundamental complex response.

The notebook reports:

- fundamental strain amplitude;
- fundamental stress amplitude;
- \(|G_1^*|\);
- stress-strain phase difference;
- approximate \(G_1'\);
- approximate \(G_1''\).

This is a direct Fourier estimate, not a replacement for the complete rheological model-fitting procedure used in the papers.


In [ ]:

def complex_fundamental_metrics(t, stress_values, strain_values, f0):
    stress_values = np.asarray(stress_values, dtype=float)
    strain_values = np.asarray(strain_values, dtype=float)

    stress_centered = stress_values - np.mean(stress_values)
    strain_centered = strain_values - np.mean(strain_values)

    dt = np.mean(np.diff(t))
    f = rfftfreq(len(t), d=dt)
    S = rfft(stress_centered)
    G = rfft(strain_centered)

    idx = int(np.argmin(np.abs(f - f0)))

    if np.abs(G[idx]) == 0:
        raise ZeroDivisionError("Fundamental strain coefficient is zero.")

    H = S[idx] / G[idx]
    phase_delta = np.angle(H)
    Gstar = np.abs(H)

    # With stress leading strain by delta:
    G_storage = Gstar * np.cos(phase_delta)
    G_loss = Gstar * np.sin(phase_delta)

    # One-sided sinusoidal amplitudes for reporting.
    stress_amp = 2 * np.abs(S[idx]) / len(t)
    strain_amp = 2 * np.abs(G[idx]) / len(t)

    return {
        "fft_frequency_hz": f[idx],
        "stress_H1_amplitude_Pa": stress_amp,
        "strain_H1_amplitude": strain_amp,
        "|G1*|_Pa": Gstar,
        "phase_stress_minus_strain_deg": np.degrees(phase_delta),
        "G1_prime_Pa_approx": G_storage,
        "G1_double_prime_Pa_approx": G_loss,
    }

if {"stress", "strain"}.issubset(uniform_block.columns):
    modulus_metrics = complex_fundamental_metrics(
        uniform_block["time"].to_numpy(),
        uniform_block["stress"].to_numpy(),
        uniform_block["strain"].to_numpy(),
        f0_for_analysis
    )
    display(pd.DataFrame([modulus_metrics]))
else:
    print("Stress and strain are both required for the complex-modulus estimate.")



## 14. Batch harmonic comparison across all LAOS experiments

This section repeats the stress FFT for every extracted periodic experiment whose protocol contains an angular frequency.

It is useful for examining how nonlinear harmonic content changes with:

- strain amplitude \(\gamma_0\);
- angular frequency \(\omega\);
- donor/dataset.


In [ ]:

def summarize_one_block(label, df, n_cycles=5, n_harmonics=9):
    info = parse_protocol_label(label)

    if not np.isfinite(info["f0_hz"]) or info["f0_hz"] <= 0:
        return None

    if "stress" not in df.columns:
        return None

    uniform, used_cycles = uniformize_block(
        df,
        f0=info["f0_hz"],
        n_cycles=n_cycles
    )

    if len(uniform) < 16:
        return None

    result = compute_fft(
        uniform["time"].to_numpy(),
        uniform["stress"].to_numpy(),
        window=FFT_WINDOW,
        detrend=DETREND
    )

    ht = harmonic_table(
        result,
        f0=info["f0_hz"],
        n_harmonics=n_harmonics
    )

    if ht.empty:
        return None

    H1 = ht.loc[ht["harmonic"] == 1, "amplitude"].iloc[0]

    row = {
        "experiment": label,
        "omega_rad_s": info["omega_rad_s"],
        "f0_hz": info["f0_hz"],
        "gamma": info["gamma"],
        "cycles_used": used_cycles,
        "samples": len(uniform),
        "H1_amplitude": H1,
    }

    for h in range(1, n_harmonics + 1):
        subset = ht.loc[ht["harmonic"] == h]
        if len(subset):
            amp = subset["amplitude"].iloc[0]
            row[f"H{h}_amplitude"] = amp
            row[f"H{h}_over_H1"] = amp / H1 if H1 > 0 else np.nan

    higher_cols = [
        f"H{h}_amplitude"
        for h in range(2, n_harmonics + 1)
        if f"H{h}_amplitude" in row
    ]
    odd_cols = [
        f"H{h}_amplitude"
        for h in range(3, n_harmonics + 1, 2)
        if f"H{h}_amplitude" in row
    ]

    row["THD"] = (
        np.sqrt(sum(row[c] ** 2 for c in higher_cols)) / H1
        if H1 > 0 and higher_cols else np.nan
    )
    row["odd_THD"] = (
        np.sqrt(sum(row[c] ** 2 for c in odd_cols)) / H1
        if H1 > 0 and odd_cols else np.nan
    )
    row["I3_over_I1"] = row.get("H3_over_H1", np.nan)

    return row

batch_rows = []
skipped = []

for label, df in blocks.items():
    try:
        row = summarize_one_block(
            label,
            df,
            n_cycles=N_CYCLES,
            n_harmonics=N_HARMONICS
        )
        if row is not None:
            batch_rows.append(row)
        else:
            skipped.append(label)
    except Exception as exc:
        skipped.append(f"{label} -> {exc}")

batch_summary = pd.DataFrame(batch_rows)

if len(batch_summary):
    batch_summary = batch_summary.sort_values(
        ["omega_rad_s", "gamma"],
        na_position="last"
    ).reset_index(drop=True)

display(batch_summary)

if skipped:
    print("\nSkipped non-periodic / unsupported blocks:")
    for item in skipped[:20]:
        print(" -", item)


In [ ]:

if len(batch_summary):
    # H1 amplitude vs strain amplitude
    fig, ax = plt.subplots(figsize=(10, 5))
    for omega, grp in batch_summary.groupby("omega_rad_s"):
        grp = grp.sort_values("gamma")
        ax.plot(
            grp["gamma"],
            grp["H1_amplitude"],
            marker="o",
            label=f"omega={omega:g} rad/s"
        )
    if batch_summary["gamma"].dropna().min() > 0 and (
        batch_summary["gamma"].dropna().max() /
        batch_summary["gamma"].dropna().min()
    ) > 20:
        ax.set_xscale("log")
    ax.set_title("First-harmonic stress amplitude vs strain amplitude")
    ax.set_xlabel("Strain amplitude gamma")
    ax.set_ylabel("H1 stress amplitude (Pa)")
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.show()

    # I3/I1 vs strain amplitude
    if "I3_over_I1" in batch_summary.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        for omega, grp in batch_summary.groupby("omega_rad_s"):
            grp = grp.sort_values("gamma")
            ax.plot(
                grp["gamma"],
                100 * grp["I3_over_I1"],
                marker="o",
                label=f"omega={omega:g} rad/s"
            )
        if batch_summary["gamma"].dropna().min() > 0 and (
            batch_summary["gamma"].dropna().max() /
            batch_summary["gamma"].dropna().min()
        ) > 20:
            ax.set_xscale("log")
        ax.set_title("Third-to-first harmonic ratio vs strain amplitude")
        ax.set_xlabel("Strain amplitude gamma")
        ax.set_ylabel("I3 / I1 (%)")
        ax.legend()
        ax.grid(True, alpha=0.25)
        plt.show()

    # THD vs strain amplitude
    fig, ax = plt.subplots(figsize=(10, 5))
    for omega, grp in batch_summary.groupby("omega_rad_s"):
        grp = grp.sort_values("gamma")
        ax.plot(
            grp["gamma"],
            100 * grp["THD"],
            marker="o",
            label=f"omega={omega:g} rad/s"
        )
    if batch_summary["gamma"].dropna().min() > 0 and (
        batch_summary["gamma"].dropna().max() /
        batch_summary["gamma"].dropna().min()
    ) > 20:
        ax.set_xscale("log")
    ax.set_title("Total harmonic distortion vs strain amplitude")
    ax.set_xlabel("Strain amplitude gamma")
    ax.set_ylabel("THD (%)")
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.show()


## 15. Export FFT results

In [ ]:

safe_experiment = re.sub(r"[^A-Za-z0-9._-]+", "_", selected_experiment).strip("_")
prefix = f"{DATASET_KEY}_{selected_sheet}_{safe_experiment}"

spectrum_export = pd.DataFrame({
    "frequency_hz": fft_result["freq"],
    "amplitude": fft_result["amplitude"],
    "phase_deg": np.degrees(fft_result["phase_rad"]),
    "relative_power_pct": fft_result["relative_power_pct"],
})

spectrum_path = RESULTS_DIR / f"{prefix}_spectrum.csv"
harmonics_path = RESULTS_DIR / f"{prefix}_harmonics.csv"
sampling_path = RESULTS_DIR / f"{prefix}_sampling_summary.csv"
batch_path = RESULTS_DIR / f"{DATASET_KEY}_{selected_sheet}_batch_harmonic_summary.csv"

spectrum_export.to_csv(spectrum_path, index=False)
harmonics.to_csv(harmonics_path, index=False)
sampling_summary.to_csv(sampling_path, index=False)

if len(batch_summary):
    batch_summary.to_csv(batch_path, index=False)

print("Saved:")
print(" -", spectrum_path)
print(" -", harmonics_path)
print(" -", sampling_path)
if len(batch_summary):
    print(" -", batch_path)



## 16. Interpretation guide

### Fundamental frequency \(H_1\)

For LAOS, the primary response should occur near

\[
f_0 = \frac{\omega}{2\pi}.
\]

The first harmonic usually contains most of the signal energy when the response is close to sinusoidal.

### Higher harmonics

A perfectly linear sinusoidal response is dominated by \(H_1\). Increasing contributions at \(H_3,H_5,\ldots\) indicate increasing nonlinear distortion.

For blood LAOS analysis, the ratio

\[
\frac{I_3}{I_1}
\]

is especially useful as a compact measure of nonlinearity.

### THD

The notebook calculates

\[
\mathrm{THD}
=
\frac{\sqrt{A_2^2+A_3^2+\cdots+A_N^2}}{A_1},
\]

and an odd-only form using \(H_3,H_5,\ldots\).

### Phase

The phase difference between fundamental stress and strain describes whether the response is more in-phase with strain or shifted toward a viscous response. The notebook also reports an FFT-based estimate of \(G_1'\) and \(G_1''\).

### Frequency resolution

The smallest resolvable FFT-bin spacing is approximately

\[
\Delta f = \frac{1}{T},
\]

where \(T\) is the analyzed time-window duration. Longer windows improve frequency resolution.

### Nyquist frequency

For sampling frequency \(f_s\),

\[
f_{\mathrm{Nyquist}}=\frac{f_s}{2}.
\]

Frequencies above Nyquist cannot be uniquely recovered from the sampled signal.

### Important limitation for changepoints

A standard FFT analyzes the **whole selected window at once**. If blood properties change halfway through a window, FFT can reveal that additional spectral content exists, but it cannot precisely tell you when the change occurred.

For real-time transition detection, use this FFT notebook as a global spectral baseline, then compare it with a time-localized method such as STFT or CWT.
